# Notebook 1 — Time-Course / Kinetic Model
### AMPK Signaling Case Study: Metformin at the Triad

The dose-response simulator treats "AMPK activation level" as a fixed, static input. In reality,
metformin plasma concentration rises and falls after each dose, AMPK activation tracks that
concentration with a short lag, mTORC1 relaxes toward a new target more slowly, and tissue-level
downstream markers (hepatic glucose output, tumor proliferation signal, placental sFlt-1) lag
furthest behind — because they depend on sustained transcriptional and secretory changes, not just
signaling-protein phosphorylation state.

This notebook builds a simple pharmacokinetic/pharmacodynamic (PK/PD) ODE model to show **onset,
peak, and decay** of effect over time, and to illustrate why the placental response in particular
requires **sustained dosing** to reach its therapeutic window — a point worth making explicit in
your write-up's discussion of clinical trial design.

**Model structure**

State variables: `C` (drug concentration, arbitrary units) → `A` (AMPK activation, 0–100) →
`M` (mTORC1 activity, 0–100) → three tissue-specific downstream markers, each relaxing toward the
steady-state target curves used in the original dose-response simulator.


In [ ]:
!pip install ipywidgets -q
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox


### Steady-state target functions (reused from the dose-response simulator)

In [ ]:
def target_liver_glucose(A):
    """Hepatic glucose output target, given AMPK activation A (0-100). Lower is better."""
    return np.clip(100 - 0.8*A, 14, 100)

def target_tumor_prolif(A):
    """Tumor proliferation-signal target. Lower is better. Saturating suppression."""
    return np.clip(100 - 100*A/(A+25), 8, 100)

def target_placenta_sflt(A):
    """Placental sFlt-1 target -- biphasic ('the paradox'). Lower is better, up to a point."""
    x = np.clip(A, 0, 100)
    return 90 - 60*np.sin(np.pi*x/100) + 25*(x/100)**4


### ODE system and dosing schedule

In [ ]:
def rhs(t, y, p, dose_rate_fn):
    C, A, M, Dl, Dt, Dp = y
    dC  = dose_rate_fn(t) - p['k_e']*C
    dA  = p['k_a']*C*(100-A) - p['k_da']*A
    M_target = 100 - p['alpha']*A
    dM  = p['k_relax_m']*(M_target - M)
    dDl = p['k_d_liver']*(target_liver_glucose(A) - Dl)
    dDt = p['k_d_tumor']*(target_tumor_prolif(A) - Dt)
    dDp = p['k_d_placenta']*(target_placenta_sflt(A) - Dp)
    return [dC, dA, dM, dDl, dDt, dDp]

def make_dosing(dose_amt, interval_h, t_end, bolus_width=0.25):
    """Approximates repeated oral dosing as short input pulses every `interval_h` hours."""
    times = np.arange(0, t_end, interval_h)
    def rate(t):
        for dt in times:
            if dt <= t < dt + bolus_width:
                return dose_amt / bolus_width
        return 0.0
    return rate

DEFAULT_PARAMS = dict(
    k_e=0.15,        # drug elimination rate
    k_a=0.02,         k_da=0.05,     # AMPK activation / deactivation rates
    alpha=0.6,        k_relax_m=0.08,# mTORC1 sensitivity & relaxation rate
    k_d_liver=0.05,   # fast transcriptional response (hours)
    k_d_tumor=0.04,   # moderate (hours-to-a-day)
    k_d_placenta=0.01 # slow -- vascular/secretory remodeling (days)
)


### Acute view: a single dosing day (0–48 h)

Notice AMPK tracks each dose pulse almost immediately, mTORC1 relaxes over a few hours, and the
placental marker (sFlt-1) barely moves in just two days — the transcriptional/secretory response is
much slower than the signaling response.

In [ ]:
def run_sim(dose_amt=40, interval_h=8, t_end=48, params=DEFAULT_PARAMS):
    dose_fn = make_dosing(dose_amt, interval_h, t_end)
    y0 = [0, 0, 100, 100, 100, 90]  # start at pre-treatment baseline (placenta baseline=90, preeclamptic)
    sol = solve_ivp(rhs, [0, t_end], y0, args=(params, dose_fn), max_step=0.1, dense_output=True)
    tt = np.linspace(0, t_end, 400)
    yy = sol.sol(tt)
    return tt, yy

def plot_sim(dose_amt=40, interval_h=8, t_end=48, title="Acute dosing (single day view)"):
    tt, yy = run_sim(dose_amt, interval_h, t_end)
    fig, axes = plt.subplots(2, 1, figsize=(9,7), sharex=True)
    axes[0].plot(tt, yy[0], label='Drug concentration (C)', color='#C9A227')
    axes[0].plot(tt, yy[1], label='AMPK activation', color='#B15A2E')
    axes[0].plot(tt, yy[2], label='mTORC1 activity', color='#6B4C9A')
    axes[0].set_ylabel('Level (0-100, C unscaled)')
    axes[0].legend(loc='upper right'); axes[0].set_title(title)

    axes[1].plot(tt, yy[3], label='Liver: glucose output', color='#1F6F6B')
    axes[1].plot(tt, yy[4], label='Tumor: proliferation signal', color='#6B3F6E')
    axes[1].plot(tt, yy[5], label='Placenta: sFlt-1', color='#B15A2E')
    axes[1].set_ylabel('Downstream marker level')
    axes[1].set_xlabel('Time (hours)')
    axes[1].legend(loc='upper right')
    plt.tight_layout(); plt.show()

plot_sim(dose_amt=40, interval_h=8, t_end=48)


### Chronic view: 30 days of dosing

Extending the simulation shows the placental marker eventually reaching its target — this is the
kinetic argument for why preeclampsia trials (e.g. metformin extended-release regimens tested over
weeks, not days) need sustained dosing periods before an effect on sFlt-1 would be expected to show
up, unlike the rapid glycemic response seen in diabetes management.

In [ ]:
plot_sim(dose_amt=40, interval_h=8, t_end=24*30, title="Chronic dosing (30-day view)")


### Interactive explorer
Adjust dose size and dosing interval and watch onset/decay shift across all three tissues.

In [ ]:
@interact(
    dose_amt=FloatSlider(min=10, max=100, step=5, value=40, description='Dose size'),
    interval_h=FloatSlider(min=2, max=24, step=1, value=8, description='Interval (h)'),
    days=IntSlider(min=1, max=30, step=1, value=3, description='Days shown')
)
def explore(dose_amt=40, interval_h=8, days=3):
    plot_sim(dose_amt=dose_amt, interval_h=interval_h, t_end=24*days,
              title=f"{days}-day view: dose={dose_amt}, every {interval_h}h")


### Discussion prompts for your write-up

- Why might a clinical trial testing metformin for preeclampsia need weeks of dosing before an
  effect on sFlt-1 is measurable, while a diabetes trial can show glycemic benefit within days?
- What would happen to the placental curve if `k_d_placenta` were tissue-specific and varied by
  gestational age or disease severity? Try changing the parameter and re-running.
- The `alpha` parameter controls how strongly AMPK suppresses mTORC1. What happens to the placental
  overshoot if you increase `alpha` — does the therapeutic window narrow or widen?
